# TrustLens — Phase J: Multimodal Statistical Anomaly Analysis

**Unsupervised Statistical Novelty & Cross-Modal Outlier Modeling**

This notebook executes deterministic Isolation Forest anomaly detection across four distinct feature spaces to study **statistical novelty** across 2,980 canonical OLX listings.

### Strict Scientific Guardrails
- **ANOMALY != FRAUD**: Statistical novelty indicates feature divergence, NOT scam probability or seller guilt.
- **Zero target leakage**: Strictly unsupervised modeling without post-hoc labels or future outcomes.
- **Reproducibility**: Fixed random seed (42), robust preprocessing, and explicit contamination sensitivity.

In [ ]:
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

from trustlens.marketplace.anomaly_detection import AnomalyAnalysisEngine
print("Libraries and AnomalyAnalysisEngine imported successfully!")

## 1. Load Frozen Phase I Feature Store
Load canonical feature store table (2,980 rows x 137 columns).

In [ ]:
engine = AnomalyAnalysisEngine(random_state=42, base_contamination=0.05)
df = engine.load_unified_features()
print(f"Loaded feature store: {df.shape[0]:,} listings x {df.shape[1]:,} features")
assert len(df) == 2980
assert df.listing_id.nunique() == 2980

## 2. Feature & Leakage Audits
Audit all 137 features, exclude constants, and verify zero target leakage.

In [ ]:
audit_df = engine.perform_feature_audit()
print(f"Audited {len(audit_df)} features")
print(f"Constants detected: {audit_df.constant_flag.sum()}")
engine.perform_leakage_audit()
print("Leakage audit passed: 0 forbidden target/fraud terms found")

## 3. Build Four Experimental Feature Matrices
Construct matrices for J_PRICE, J_PRICE_TEXT, J_PRICE_IMAGE, and J_FULL_MULTIMODAL.

In [ ]:
matrices = engine.build_experiment_matrices()
for k, m in matrices.items():
    print(f"{k:20s}: {m.shape[1]:2d} features")

## 4. Run Isolation Forest Across Experiments
Fit deterministic models and calculate anomaly scores (-decision_function).

In [ ]:
results_df = engine.run_experiments()
for prefix in ["price", "price_text", "price_image", "full_multimodal"]:
    flag_col = f"{prefix}_anomaly_flag"
    score_col = f"{prefix}_anomaly_score"
    print(f"{prefix:16s}: {results_df[flag_col].sum():3d} anomalies ({results_df[flag_col].mean()*100:.2f}%) | score range: [{results_df[score_col].min():.4f}, {results_df[score_col].max():.4f}]")

## 5. Anomaly Persistence & Overlap
Analyze listings that remain anomalous across multiple independent feature representations.

In [ ]:
pers = engine.persistence_df
print(pers.experiment_count.value_counts().sort_index())
print(f"Persistent across >= 2 experiments: {pers.persistent_2.sum():,} ({pers.persistent_2.mean()*100:.1f}%)")
print(f"Persistent across all 4 experiments:  {pers.persistent_4.sum():,} ({pers.persistent_4.mean()*100:.1f}%)")

## 6. Contamination Sensitivity
Evaluate anomaly set stability across contamination values [0.01, 0.02, 0.05, 0.10].

In [ ]:
sens = engine.run_contamination_sensitivity()
print(pd.DataFrame(sens["summary"]))

## 7. Exported Anomaly Tables Verification
Verify Parquet exports on disk.

In [ ]:
p_res = pq.read_table("data/olx_processed/anomaly_results.parquet")
p_pers = pq.read_table("data/olx_processed/anomaly_persistence.parquet")
p_aud = pq.read_table("data/olx_processed/phase_j_feature_audit.parquet")
print(f"anomaly_results.parquet:     {p_res.num_rows:,} rows x {p_res.num_columns:,} cols")
print(f"anomaly_persistence.parquet: {p_pers.num_rows:,} rows x {p_pers.num_columns:,} cols")
print(f"phase_j_feature_audit.parquet: {p_aud.num_rows:,} rows x {p_aud.num_columns:,} cols")